# Ti Adatom on a 5×5 TiN Island on TiN(001)

## 0. Introduction

This tutorial builds the cell of the Ti adatom descent study in the publication:

> **D. G. Sangiovanni, A. B. Mei, D. Edström, L. Hultman, V. Chirita, I. Petrov, and J. E. Greene**, "Effects of surface vibrations on interlayer mass transport: Ab initio molecular dynamics investigation of Ti adatom descent pathways and rates from TiN/TiN(001) islands", Physical Review B 97, 035406 (2018). [DOI: 10.1103/PhysRevB.97.035406](https://doi.org/10.1103/PhysRevB.97.035406)

The cell (Sec. II.A, Fig. 1b) is a 5×5-atom TiN island, one plane high with Ti corners, on a 12×12×3-atom
TiN(001) slab with 15.3 Å of vacuum: 457 atoms. A Ti adatom is added at each of the sites a, c and i of Fig. 6,
giving three 458-atom cells.

## 1. Prepare the Environment
### 1.1. Set up defect parameters

In [ ]:
MATERIAL_NAME = "Titanium Nitride"
MILLER_INDICES = (0, 0, 1)
# Thickness counts conventional layers of two (001) planes; the bottom plane is removed after the
# island is added, leaving the paper's three planes.
SLAB_THICKNESS = 2
VACUUM = 15.3  # Angstrom
XY_SUPERCELL_MATRIX = [[6, 0], [0, 6]]

NUMBER_OF_ADDED_LAYERS = 0.5
# x and y minima one site (1/12 of the 6×6 cell) apart put Ti atoms at the island corners
ISLAND_MIN_COORDINATE = [0.3233, 0.24, 0]  # crystal coordinates
ISLAND_MAX_COORDINATE = [0.7233, 0.64, 1]

ADATOM_ELEMENT = "Ti"

ISLAND_MATERIAL_NAME = "TiN(001) 12x12x3 island 5x5"
ADATOM_MATERIAL_NAMES = {
    "a": "TiN(001) 12x12x3 island 5x5 + Ti a (FFH island)",
    "c": "TiN(001) 12x12x3 island 5x5 + Ti c (atop-N edge)",
    "i": "TiN(001) 12x12x3 island 5x5 + Ti i (atop-N terrace)",
}

### 1.2. Install Packages
The step executes only in Pyodide environment. For other environments, the packages should be installed via `pip install` (see [README](../../README.ipynb)).

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|specific_examples")

### 1.3. Load input material

In [ ]:
from mat3ra.made.material import Material
from mat3ra.standata.materials import Materials

material = Material.create(Materials.get_by_name_first_match(MATERIAL_NAME))

### 1.4. Create and preview Slab

In [ ]:
from mat3ra.made.tools.helpers import create_slab
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials

slab = create_slab(
    crystal=material,
    termination_top=None,  # Use default termination
    miller_indices=MILLER_INDICES,
    number_of_layers=SLAB_THICKNESS,
    vacuum=VACUUM,
    use_orthogonal_c=True,
    xy_supercell_matrix=XY_SUPERCELL_MATRIX
)

visualize_materials(slab)

## 2. Create the Island

In [ ]:
import numpy as np
from mat3ra.made.tools.analyze.other import get_atom_indices_by_layer
from mat3ra.made.tools.convert import to_ase
from mat3ra.made.tools.entities.coordinate import BoxCoordinateCondition
from mat3ra.made.tools.helpers import create_island
from mat3ra.made.tools.modify import filter_by_ids, remove_vacuum

coordinate_condition = BoxCoordinateCondition(min_coordinate=ISLAND_MIN_COORDINATE, max_coordinate=ISLAND_MAX_COORDINATE)

slab_with_island = create_island(
    slab=slab,
    condition=coordinate_condition,
    number_of_added_layers=NUMBER_OF_ADDED_LAYERS,
    use_cartesian_coordinates=False,
)
bottom_plane = get_atom_indices_by_layer(slab_with_island)[0]
slab_with_island = filter_by_ids(slab_with_island, ids=bottom_plane, invert=True, reset_ids=True)
slab_with_island = remove_vacuum(slab_with_island, fixed_padding=VACUUM)
slab_with_island.name = ISLAND_MATERIAL_NAME

atoms = to_ase(slab_with_island)
symbols = np.array(atoms.get_chemical_symbols())
planes = get_atom_indices_by_layer(slab_with_island)
island, terrace = planes[-1], planes[-2]
island_x, island_y = atoms.positions[island, 0], atoms.positions[island, 1]
is_corner = ((np.isclose(island_x, island_x.min()) | np.isclose(island_x, island_x.max()))
             & (np.isclose(island_y, island_y.min()) | np.isclose(island_y, island_y.max())))
print(f"{slab_with_island.name}: {len(atoms)} atoms ({sum(symbols == 'Ti')} Ti, {sum(symbols == 'N')} N) "
      f"in {len(planes)} planes, c = {atoms.cell[2, 2]:.3f} Å")
print(f"Island: {len(island)} atoms, corners {symbols[island][is_corner].tolist()}")

## 3. Add the Adatom at the Sites of Fig. 6

The paper gives no coordinates for the sites; they follow from the lattice and are measured on the island
built above. Take the N atom with the smaller x in the island's edge row at the largest y, and the bulk site
spacing d = a/2:

- **a**, the fourfold hollow between that N and the edge-centre Ti, the paper's FFH_edge:
  (x_N + d/2, y_edge − d/2), d above the island plane;
- **c**: (x_N, y_edge), d above the island plane;
- **i**: (x_N, y_edge + d), d above the terrace.

In [ ]:
from mat3ra.made.tools.helpers import create_defect_point_interstitial

site_spacing = material.lattice.a / 2
edge_y = island_y.max()
edge_nitrogen_x = min(atoms.positions[index, 0] for index in island
                      if symbols[index] == "N" and np.isclose(atoms.positions[index, 1], edge_y))
island_z = atoms.positions[island, 2].mean()
terrace_z = atoms.positions[terrace, 2].mean()

adatom_sites = {
    "a": [edge_nitrogen_x + site_spacing / 2, edge_y - site_spacing / 2, island_z + site_spacing],
    "c": [edge_nitrogen_x, edge_y, island_z + site_spacing],
    "i": [edge_nitrogen_x, edge_y + site_spacing, terrace_z + site_spacing],
}

materials_with_adatom = []
for site, coordinate in adatom_sites.items():
    material_with_adatom = create_defect_point_interstitial(
        slab_with_island, coordinate, ADATOM_ELEMENT, "exact_coordinate", use_cartesian_coordinates=True
    )
    material_with_adatom.name = ADATOM_MATERIAL_NAMES[site]
    materials_with_adatom.append(material_with_adatom)

    atoms_with_adatom = to_ase(material_with_adatom)
    adatom = len(atoms_with_adatom) - 1
    distances = np.sort(atoms_with_adatom.get_distances(adatom, range(adatom), mic=True))[:3]
    print(f"{material_with_adatom.name}: {len(atoms_with_adatom)} atoms, "
          f"adatom's three shortest distances {np.round(distances, 3)} Å")

## 4. Visualize the Result(s)

In [ ]:
visualize_materials([{"material": material_to_show, "title": material_to_show.name, "rotation": "-45x"}
                     for material_to_show in [slab_with_island, *materials_with_adatom]])

## 5. Pass data to the outside runtime

In [ ]:
from mat3ra.notebooks_utils.io import download_content_to_file
from mat3ra.notebooks_utils.material import set_materials

materials = [slab_with_island, *materials_with_adatom]
set_materials(materials)
for material_to_save in materials:
    download_content_to_file(material_to_save.to_json(), f"{material_to_save.name}.json")